# Outlook

In this notebook we code **Truncated Quantile Critics (TQC)**, described in
[this paper](https://arxiv.org/abs/2005.04269). TQC extends SAC (`08`) in one
way: instead of predicting a single Q-value, each critic predicts a
**distribution** of returns as a set of *quantiles*; the target then **drops
the top quantiles** (the optimistic tail) to control the overestimation bias
more finely than the min-of-two-critics trick of TD3/SAC.

Everything else — the squashed Gaussian actor, the entropy tuning, the replay
buffer — is unchanged from SAC.

## Setup

**On Colab**: run the installation cell below (generated automatically,
pinned versions). **Locally**: install the environment with `uv sync`, no
need for this cell.

In [ ]:
import copy
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import rl_mind.envs
from dataclasses import dataclass, replace
from torch import Tensor
from torch.distributions import Independent, Normal, TanhTransform, TransformedDistribution
from torch.utils.tensorboard import SummaryWriter
from tqdm.auto import tqdm
from rl_mind.profiles import Profile
from rl_mind.core import Actor, StochasticAction
from rl_mind.nn import build_mlp, soft_update
from rl_mind.env import VecEnv
from rl_mind.data import ReplayBuffer, Transitions
from rl_mind.collectors import TransitionCollector
from rl_mind.evaluation import Evaluator
from rl_mind.notebook import run_directory, setup_tensorboard, silence_known_warnings


In [ ]:
# pygame and tensorboard still use the deprecated `pkg_resources` API
silence_known_warnings()

# The Squashed Gaussian actor

Identical to SAC: a Gaussian squashed through a $\tanh$, sampled with the
reparametrization trick.

In [ ]:
class SquashedGaussianActor(Actor[StochasticAction]):
    def __init__(
        self,
        obs_dim: int,
        hidden: tuple[int, ...],
        action_dim: int,
        min_std: float = 1e-4,
    ):
        super().__init__()
        self.backbone = build_mlp([obs_dim, *hidden], output_activation=nn.ReLU())
        self.mean_layer = nn.Linear(hidden[-1], action_dim)
        self.std_layer = nn.Linear(hidden[-1], action_dim)
        self.softplus = nn.Softplus()
        self.min_std = min_std
        self.tanh = TanhTransform(cache_size=1)

    def dist(self, obs: Tensor) -> TransformedDistribution:
        features = self.backbone(obs)
        std = self.softplus(self.std_layer(features)) + self.min_std
        normal = Independent(Normal(self.mean_layer(features), std), 1)
        return TransformedDistribution(normal, [self.tanh])

    def forward(self, obs: Tensor) -> StochasticAction:
        dist = self.dist(obs)
        action = dist.rsample()
        log_prob = dist.log_prob(action)
        self.tanh._cached_x_y = [None, None]
        return StochasticAction(value=action, log_prob=log_prob)

    def act(self, obs: Tensor) -> Tensor:
        features = self.backbone(obs)
        return torch.tanh(self.mean_layer(features))

# The quantile critic

The critic is a set of `n_nets` networks, each outputting `n_quantiles`
values — an approximation of the *distribution* of the return $Q(s, a)$ by
its quantiles. `forward` stacks them into a `[B, n_nets, n_quantiles]`
tensor.

In [ ]:
class TruncatedQuantileCritic(nn.Module):
    def __init__(
        self,
        obs_dim: int,
        hidden: tuple[int, ...],
        action_dim: int,
        n_nets: int,
        n_quantiles: int,
    ):
        super().__init__()
        self.nets = nn.ModuleList(
            build_mlp([obs_dim + action_dim, *hidden, n_quantiles])
            for _ in range(n_nets)
        )

    def forward(self, obs: Tensor, action: Tensor) -> Tensor:
        obs_act = torch.cat([obs, action], dim=1)
        return torch.stack([net(obs_act) for net in self.nets], dim=1)

# The quantile Huber loss

The critic regresses its quantiles towards the target quantiles using the
quantile Huber loss (from
[QR-DQN](https://arxiv.org/abs/1710.10044)): a Huber loss weighted
asymmetrically so that the $i$-th predicted quantile converges to the
corresponding quantile of the target distribution. This part is provided.

In [ ]:
def quantile_huber_loss(quantiles: Tensor, target: Tensor) -> Tensor:
    """Quantile Huber loss between predicted quantiles and target samples

    :param quantiles: The predicted quantiles, shape `[B, n_nets, n_quantiles]`
    :param target: The target samples, shape `[B, n_targets]`
    """
    # pairwise differences: [B, n_nets, n_quantiles, n_targets]
    pairwise = target[:, None, None, :] - quantiles[:, :, :, None]
    abs_pairwise = pairwise.abs()
    huber = torch.where(abs_pairwise > 1, abs_pairwise - 0.5, 0.5 * pairwise**2)

    n_quantiles = quantiles.shape[2]
    tau = (torch.arange(n_quantiles).float() + 0.5) / n_quantiles
    weight = torch.abs(tau[None, None, :, None] - (pairwise < 0).float())
    return (weight * huber).mean()

# The critic loss

The target pools the quantiles predicted by **all** the target nets at the
next state (for an action sampled from the current actor), sorts them, and
**drops the `top_quantiles_to_drop` per net** — the truncation that removes
the optimistic tail. As in SAC, the (entropy-augmented) target is

$$ y = r + \gamma\,(1-\text{terminated})\,\big(z - \alpha \log\pi(a'|s')\big) $$

for each kept quantile $z$.

In [ ]:
def compute_critic_loss(
    gamma: float,
    alpha: float,
    top_quantiles_to_drop: int,
    batch: Transitions[StochasticAction],
    actor: SquashedGaussianActor,
    critic: TruncatedQuantileCritic,
    target_critic: TruncatedQuantileCritic,
) -> Tensor:
    """Compute the TQC critic loss"""
    quantiles = critic(batch.obs, batch.action.value)  # [B, n_nets, n_quantiles]
    n_nets = quantiles.shape[1]

    # Compute the truncated, entropy-augmented target quantiles

    with torch.no_grad():
        next_action = actor(batch.next_obs)          # current actor at s'
        next_quantiles = target_critic(...)          # [B, n_nets, n_quantiles]
        # pool all nets, sort, drop the top `top_quantiles_to_drop * n_nets`
        sorted_q, _ = next_quantiles.reshape(quantiles.shape[0], -1).sort(dim=1)
        n_keep = sorted_q.shape[1] - top_quantiles_to_drop * n_nets
        truncated = sorted_q[:, :n_keep]
        target = ...                                 # [B, n_keep]
    assert False, 'Not implemented yet'


    return quantile_huber_loss(quantiles, target)

# The actor loss

The actor maximizes the (entropy-augmented) **mean** of its quantiles — the
distributional analogue of SAC's actor loss.

In [ ]:
def compute_actor_loss(
    alpha: float,
    obs: Tensor,
    actor: SquashedGaussianActor,
    critic: TruncatedQuantileCritic,
) -> tuple[Tensor, Tensor]:
    """Compute the TQC actor loss and return the sampled log-probabilities"""
    # Compute the actor loss

    action = actor(obs)
    mean_q = critic(obs, action.value).mean(dim=(1, 2))   # mean over nets and quantiles
    actor_loss = ...
    assert False, 'Not implemented yet'

    return actor_loss, action.log_prob

## Hyper-parameters

In [ ]:
@dataclass(frozen=True)
class TQCConfig:
    env_name: str = "CartPoleContinuous-v1"
    seed: int = 1

    max_steps: int = Profile.pick(30_000, fast_test=1_200)
    n_envs: int = 8
    steps_per_round: int = 32
    updates_per_round: int = 32
    learning_starts: int = Profile.pick(10_000, fast_test=300)

    buffer_size: int = 1_000_000
    batch_size: int = 256

    gamma: float = 0.98
    tau: float = 0.05
    max_grad_norm: float = 0.5

    entropy_mode: str = "auto"
    init_entropy_coef: float = 0.2

    #: distributional critic: number of nets, quantiles, and how many top
    #: quantiles to drop per net (the truncation)
    n_nets: int = 2
    n_quantiles: int = 25
    top_quantiles_to_drop: int = 2

    actor_hidden: tuple[int, ...] = (256, 256)
    critic_hidden: tuple[int, ...] = (256, 256)
    lr: float = 3e-4

    eval_interval: int = Profile.pick(2_000, fast_test=400)
    n_eval_envs: int = 10

## Main training loop

Identical in structure to SAC — only the critic and its loss changed.

In [ ]:
def run_tqc(cfg: TQCConfig) -> Evaluator:
    torch.manual_seed(cfg.seed)
    env = VecEnv(cfg.env_name, cfg.n_envs, seed=cfg.seed)

    actor = SquashedGaussianActor(env.observation_dim, cfg.actor_hidden, env.action_dim)
    critic = TruncatedQuantileCritic(
        env.observation_dim,
        cfg.critic_hidden,
        env.action_dim,
        cfg.n_nets,
        cfg.n_quantiles,
    )
    target_critic = copy.deepcopy(critic)

    actor_optimizer = torch.optim.Adam(actor.parameters(), lr=cfg.lr)
    critic_optimizer = torch.optim.Adam(critic.parameters(), lr=cfg.lr)

    auto_entropy = cfg.entropy_mode == "auto"
    log_alpha = torch.log(torch.ones(1) * cfg.init_entropy_coef).requires_grad_(
        auto_entropy
    )
    alpha_optimizer = torch.optim.Adam([log_alpha], lr=cfg.lr) if auto_entropy else None
    target_entropy = -float(env.action_dim)

    collector = TransitionCollector(env, actor)
    buffer = ReplayBuffer(cfg.buffer_size)
    run_dir = run_directory(f"tqc-{cfg.env_name}-S{cfg.seed}")
    writer = SummaryWriter(run_dir)
    evaluator = Evaluator(
        VecEnv(cfg.env_name, cfg.n_eval_envs, seed=cfg.seed + 100),
        every=cfg.eval_interval,
        run_dir=run_dir,
        writer=writer,
    )

    pbar = tqdm(total=cfg.max_steps)
    while collector.steps < cfg.max_steps:
        buffer.add(collector.collect(cfg.steps_per_round))
        pbar.update(collector.steps - pbar.n)
        if len(buffer) < cfg.learning_starts:
            continue

        for _ in range(cfg.updates_per_round):
            batch = buffer.sample(cfg.batch_size)
            alpha = log_alpha.exp().item()

            critic_loss = compute_critic_loss(
                cfg.gamma,
                alpha,
                cfg.top_quantiles_to_drop,
                batch,
                actor,
                critic,
                target_critic,
            )
            critic_optimizer.zero_grad()
            critic_loss.backward()
            critic_optimizer.step()

            actor_loss, log_prob = compute_actor_loss(alpha, batch.obs, actor, critic)
            actor_optimizer.zero_grad()
            actor_loss.backward()
            actor_optimizer.step()

            if auto_entropy:
                alpha_loss = -(log_alpha * (log_prob + target_entropy).detach()).mean()
                alpha_optimizer.zero_grad()
                alpha_loss.backward()
                alpha_optimizer.step()

            soft_update(critic, target_critic, cfg.tau)

        writer.add_scalar("loss/critic", critic_loss.item(), collector.steps)
        writer.add_scalar("loss/actor", actor_loss.item(), collector.steps)
        writer.add_scalar("entropy_coef", alpha, collector.steps)
        if result := evaluator.run_if_needed(collector.steps, actor):
            pbar.set_description(
                f"eval={result.mean:7.1f} best={evaluator.best_reward:7.1f}"
            )

    pbar.close()
    return evaluator

## Launching tensorboard to visualize the results

In [ ]:
setup_tensorboard()

In [ ]:
cfg = TQCConfig()
tqc = run_tqc(cfg)

In [ ]:
tqc.visualize_best()

## Learning curve

In [ ]:
plt.plot([r.step for r in tqc.history], [r.mean for r in tqc.history])
plt.xlabel("steps")
plt.ylabel("cumulated reward")
plt.title(f"TQC on {cfg.env_name}")
plt.show()

## Exercise

Vary `top_quantiles_to_drop`: with 0 dropped quantiles TQC is optimistic
(overestimates), while dropping too many makes it overly pessimistic and
slow. Find the sweet spot for `CartPoleContinuous-v1`.